# Built-in vs Custom PTQ Accuracy on Trained ResNet-18



In [14]:
from pathlib import Path
import json
import sys

import keras_hub
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import inspect_tflite
from src.models.cnn_based.pretrained_resnet18 import CIFAR10_CLASS_NAMES
from src.quantization.custom_quantization import custom_ptq, dequantize_tensor
from src.quantization.inbuilt_quantization import (
    build_fixed_input_model,
    convert_full_integer,
)

## 1. Configuration

By default the final comparison uses all 10,000 CIFAR-10 test images. Set `NUM_EVALUATION_SAMPLES` to a smaller integer for a quick smoke test.

In [15]:
MODEL_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "resnet18_cifar10_training"
    / "resnet18_cifar10_fp32.keras"
)
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "resnet18_ptq_accuracy_notebook"
BUILTIN_MODEL_PATH = OUTPUT_DIR / "resnet18_cifar10_builtin_full_int8.tflite"

NUM_CALIBRATION_SAMPLES = 100
NUM_EVALUATION_SAMPLES = None  # None evaluates all 10,000 test images.
BATCH_SIZE = 32
FORCE_RECONVERT_BUILTIN = False

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing trained model: {MODEL_PATH}. Run notebook 06 first."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = keras.models.load_model(MODEL_PATH, compile=False)
print(f"Loaded trained model: {MODEL_PATH}")
model.summary(expand_nested=True)

Loaded trained model: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_cifar10_training/resnet18_cifar10_fp32.keras


Model: "resnet18_cifar10_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ images (InputLayer)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_net_backbone                │ (None, 7, 7, 512)      │    11,186,112 │
│ (ResNetBackbone)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ input_layer_3 (InputLayer) │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_pad (ZeroPadding2D)  │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_conv (Conv2D)        │ (None, None, None, 64) │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_bn                   │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_relu (Activation)    │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ pool1_pad (ZeroPadding2D)  │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ pool1_pool (MaxPooling2D)  │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_1_conv       │ (None, None, None, 64) │        36,864 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_1_bn         │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_1_relu       │ (None, None, None, 64) │             0 │
│ (Activation)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_2_conv       │ (None, None, None, 64) │        36,864 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_2_bn         │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_add (Add)    │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_out          │ (None, None, None, 64) │             0 │
│ (Activation)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block1_1_conv       │ (None, None, None, 64) │        36,864 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block1_1_bn         │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block1_1_relu       │ (None, None, None, 64) │             0 │
│ (Activation)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 11,191,242 (42.69 MB)

 Trainable params: 11,181,642 (42.65 MB)

 Non-trainable params: 9,600 (37.50 KB)

## 2. Load CIFAR-10 and reproduce training preprocessing

Notebook 06 used the ResNet-18 ImageNet preset converter: bicubic resize to 224×224 followed by channel-wise scale and offset normalization. The constants below are copied from that preset so loading the saved classifier does not require rebuilding the ImageNet model.

In [16]:
(train_images, train_labels), (test_images, test_labels) = (
    keras.datasets.cifar10.load_data()
)
train_labels = train_labels.reshape(-1)
test_labels = test_labels.reshape(-1)

if NUM_EVALUATION_SAMPLES is not None:
    test_images = test_images[:NUM_EVALUATION_SAMPLES]
    test_labels = test_labels[:NUM_EVALUATION_SAMPLES]

image_converter = keras_hub.layers.ResNetImageConverter(
    image_size=(224, 224),
    scale=[0.017124753831663668, 0.01750700280112045, 0.017429193899782133],
    offset=[-2.1179039301310043, -2.0357142857142856, -1.8044444444444445],
    interpolation="bicubic",
    crop_to_aspect_ratio=True,
)

def preprocess_images(images):
    return image_converter(images)

calibration_samples = [
    preprocess_images(train_images[index : index + 1]).numpy()
    for index in range(NUM_CALIBRATION_SAMPLES)
]

print(f"Evaluation images: {len(test_images):,}")
print(f"Calibration images: {len(calibration_samples):,}")
print(f"Model output classes: {model.output_shape[-1]}")

Evaluation images: 10,000
Calibration images: 100
Model output classes: 10


## 3. Convert the trained model with built-in full-integer PTQ

In [17]:
def representative_dataset():
    for sample in calibration_samples:
        yield [sample]

if FORCE_RECONVERT_BUILTIN or not BUILTIN_MODEL_PATH.exists():
    fixed_model = build_fixed_input_model(model, input_shape=(224, 224, 3))
    BUILTIN_MODEL_PATH.write_bytes(
        convert_full_integer(fixed_model, representative_dataset)
    )

print(f"Built-in INT8 model: {BUILTIN_MODEL_PATH}")
print(f"Serialized size: {BUILTIN_MODEL_PATH.stat().st_size / 1024**2:.2f} MiB")

Built-in INT8 model: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_ptq_accuracy_notebook/resnet18_cifar10_builtin_full_int8.tflite
Serialized size: 10.80 MiB


## 4. Apply custom per-channel INT8 weight PTQ

`custom_ptq` stores the actual signed integer weight arrays and their per-output-channel scales. For classification evaluation only, the equation `real = scale × integer` reconstructs those fixed weights into a cloned Keras model. Rank-one bias and batch-normalization tensors remain at their original precision, matching the current custom converter's documented scope.

In [18]:
custom_results = custom_ptq(
    model,
    representative_samples=calibration_samples,
    quantize_min_rank=2,
    per_channel=True,
)
weight_result = custom_results["weights"]

quantized_tensor_iter = iter(weight_result.tensors)
reconstructed_weights = []
for weight in model.weights:
    weight_array = weight.numpy()
    should_quantize = (
        np.issubdtype(weight_array.dtype, np.floating)
        and weight_array.ndim >= 2
    )
    if should_quantize:
        quantized_tensor = next(quantized_tensor_iter)
        reconstructed_weights.append(
            dequantize_tensor(quantized_tensor).astype(weight_array.dtype)
        )
    else:
        reconstructed_weights.append(weight_array)

if list(quantized_tensor_iter):
    raise RuntimeError("Not all custom quantized tensors were consumed.")

custom_model = keras.models.clone_model(model)
custom_model.set_weights(reconstructed_weights)

pd.DataFrame([{
    "quantized_weight_tensors": len(weight_result.tensors),
    "fp32_weight_mib": weight_result.fp32_size_bytes / 1024**2,
    "custom_weight_mib": weight_result.int8_size_bytes / 1024**2,
    "compression_ratio": weight_result.compression_ratio,
    "memory_reduction_percent": weight_result.memory_reduction_percent,
}])

,quantized_weight_tensors,fp32_weight_mib,custom_weight_mib,compression_ratio,memory_reduction_percent
0,21,42.6912,10.72776,3.979507,74.871261


## 5. Tensor, value-count, and parameter-memory analysis

The original/custom counts refer to stored Keras parameter tensors. TensorFlow Lite folds 80 BatchNorm tensors into the convolution kernels and INT32 biases, so the built-in representation does not have a one-to-one correspondence with the original 102 tensors. `graph_int8_tensors` also includes the model input, intermediate activations, and output.

Memory below compares raw parameter/constant-buffer storage. Custom quantizer scales and TFLite FlatBuffer metadata are not included, keeping the comparison focused on tensor data.

In [19]:
original_parameter_tensors = len(model.weights)
original_parameter_values = int(sum(weight.numpy().size for weight in model.weights))
original_parameter_bytes = int(sum(weight.numpy().nbytes for weight in model.weights))

custom_int8_weight_tensors = len(weight_result.tensors)
custom_int8_weight_values = int(
    sum(tensor.values.size for tensor in weight_result.tensors)
)
custom_fp32_parameter_tensors = (
    original_parameter_tensors - custom_int8_weight_tensors
)
custom_parameter_bytes = int(weight_result.quantized_size_bytes)

builtin_interpreter = tf.lite.Interpreter(model_path=str(BUILTIN_MODEL_PATH))
builtin_interpreter.allocate_tensors()
builtin_tensor_details = {
    int(detail["index"]): detail
    for detail in builtin_interpreter.get_tensor_details()
}
builtin_operations = builtin_interpreter._get_ops_details()
builtin_weight_indices = set()
builtin_bias_indices = set()
for operation in builtin_operations:
    if operation["op_name"] in {"CONV_2D", "FULLY_CONNECTED"}:
        builtin_weight_indices.add(int(operation["inputs"][1]))
        builtin_bias_indices.add(int(operation["inputs"][2]))

builtin_weight_values = int(sum(
    np.prod(builtin_tensor_details[index]["shape"], dtype=np.int64)
    for index in builtin_weight_indices
))
builtin_bias_values = int(sum(
    np.prod(builtin_tensor_details[index]["shape"], dtype=np.int64)
    for index in builtin_bias_indices
))
builtin_graph_int8_tensors = sum(
    detail["dtype"] == np.int8 for detail in builtin_tensor_details.values()
)
builtin_graph_int32_tensors = sum(
    detail["dtype"] == np.int32 for detail in builtin_tensor_details.values()
)
builtin_storage = inspect_tflite(BUILTIN_MODEL_PATH)
builtin_parameter_bytes = int(builtin_storage["weight_storage_bytes"])

tensor_analysis_table = pd.DataFrame([
    {
        "model": "Original FP32 Keras",
        "stored_parameter_tensors": original_parameter_tensors,
        "int8_weight_tensors": 0,
        "int32_bias_tensors": 0,
        "fp32_parameter_tensors": original_parameter_tensors,
        "total_graph_tensors": None,
        "graph_int8_tensors": 0,
        "graph_int32_tensors": 0,
        "total_parameter_values": original_parameter_values,
        "int8_weight_values": 0,
        "int32_bias_values": 0,
    },
    {
        "model": "Custom per-channel INT8 weights",
        "stored_parameter_tensors": original_parameter_tensors,
        "int8_weight_tensors": custom_int8_weight_tensors,
        "int32_bias_tensors": 0,
        "fp32_parameter_tensors": custom_fp32_parameter_tensors,
        "total_graph_tensors": None,
        "graph_int8_tensors": 0,
        "graph_int32_tensors": 0,
        "total_parameter_values": original_parameter_values,
        "int8_weight_values": custom_int8_weight_values,
        "int32_bias_values": 0,
    },
    {
        "model": "Built-in full INT8 TFLite",
        "stored_parameter_tensors": (
            len(builtin_weight_indices) + len(builtin_bias_indices)
        ),
        "int8_weight_tensors": len(builtin_weight_indices),
        "int32_bias_tensors": len(builtin_bias_indices),
        "fp32_parameter_tensors": 0,
        "total_graph_tensors": len(builtin_tensor_details),
        "graph_int8_tensors": builtin_graph_int8_tensors,
        "graph_int32_tensors": builtin_graph_int32_tensors,
        "total_parameter_values": builtin_weight_values + builtin_bias_values,
        "int8_weight_values": builtin_weight_values,
        "int32_bias_values": builtin_bias_values,
    },
])

memory_analysis_table = pd.DataFrame([
    {
        "model": "Original FP32 Keras",
        "parameter_storage_bytes": original_parameter_bytes,
        "parameter_storage_mib": original_parameter_bytes / 1024**2,
        "compression_ratio_vs_fp32": 1.0,
        "memory_reduction_percent": 0.0,
    },
    {
        "model": "Custom per-channel INT8 weights",
        "parameter_storage_bytes": custom_parameter_bytes,
        "parameter_storage_mib": custom_parameter_bytes / 1024**2,
        "compression_ratio_vs_fp32": (
            original_parameter_bytes / custom_parameter_bytes
        ),
        "memory_reduction_percent": (
            1 - custom_parameter_bytes / original_parameter_bytes
        ) * 100,
    },
    {
        "model": "Built-in full INT8 TFLite",
        "parameter_storage_bytes": builtin_parameter_bytes,
        "parameter_storage_mib": builtin_parameter_bytes / 1024**2,
        "compression_ratio_vs_fp32": (
            original_parameter_bytes / builtin_parameter_bytes
        ),
        "memory_reduction_percent": (
            1 - builtin_parameter_bytes / original_parameter_bytes
        ) * 100,
    },
])

display(tensor_analysis_table)
display(memory_analysis_table)

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,stored_parameter_tensors,int8_weight_tensors,int32_bias_tensors,fp32_parameter_tensors,total_graph_tensors,graph_int8_tensors,graph_int32_tensors,total_parameter_values,int8_weight_values,int32_bias_values
0,Original FP32 Keras,102,0,0,102,NaN,0,0,11191242,0,0
1,Custom per-channel INT8 weights,102,21,0,81,NaN,0,0,11191242,11172032,0
2,Built-in full INT8 TFLite,42,21,21,0,82.0,58,24,11176842,11172032,4810


,model,parameter_storage_bytes,parameter_storage_mib,compression_ratio_vs_fp32,memory_reduction_percent
0,Original FP32 Keras,44764968,42.691200,1.000000,0.000000
1,Custom per-channel INT8 weights,11248872,10.727760,3.979507,74.871261
2,Built-in full INT8 TFLite,11191344,10.672897,3.999964,74.999772


### Parameter tensors grouped by rank

Custom PTQ uses `rank >= quantize_min_rank`. With the default `quantize_min_rank=2`, rank-2 Dense kernels and rank-4 Conv2D kernels are quantized; rank-1 biases and BatchNorm parameters remain FP32.

In [20]:
def count_tensors_by_rank(model, quantize_min_rank=2):
    rows = []
    for weight in model.weights:
        values = weight.numpy()
        is_floating = np.issubdtype(values.dtype, np.floating)
        rows.append({
            "tensor": getattr(weight, "path", weight.name),
            "rank": values.ndim,
            "shape": tuple(values.shape),
            "number_of_values": values.size,
            "quantized_by_custom_ptq": (
                is_floating and values.ndim >= quantize_min_rank
            ),
        })

    tensor_table = pd.DataFrame(rows)
    rank_table = (
        tensor_table.groupby("rank", as_index=False)
        .agg(
            total_tensors=("tensor", "count"),
            total_values=("number_of_values", "sum"),
            tensors_quantized=("quantized_by_custom_ptq", "sum"),
        )
        .sort_values("rank")
    )
    rank_table["tensors_not_quantized"] = (
        rank_table["total_tensors"] - rank_table["tensors_quantized"]
    )
    return rank_table, tensor_table

rank_summary, individual_tensor_ranks = count_tensors_by_rank(
    model, quantize_min_rank=2
)
rank_summary

,rank,total_tensors,total_values,tensors_quantized,tensors_not_quantized
0,1,81,19210,0,81
1,2,1,5120,1,0
2,4,20,11166912,20,0


## 6. Shared prediction helpers

In [21]:
def predict_keras_labels(keras_model, raw_images, batch_size=BATCH_SIZE):
    predictions = []
    for start in range(0, len(raw_images), batch_size):
        batch = preprocess_images(raw_images[start : start + batch_size])
        logits = keras_model(batch, training=False).numpy()
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)

def quantize_tflite_input(values, input_detail):
    scale, zero_point = input_detail["quantization"]
    if not scale:
        raise ValueError("TFLite input quantization scale must be non-zero.")
    limits = np.iinfo(input_detail["dtype"])
    quantized = np.rint(values / scale + zero_point)
    return np.clip(quantized, limits.min, limits.max).astype(
        input_detail["dtype"]
    )

def predict_tflite_labels(model_path, raw_images, batch_size=BATCH_SIZE):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    predictions = []

    for start in range(0, len(raw_images), batch_size):
        batch = preprocess_images(raw_images[start : start + batch_size]).numpy()
        input_detail = interpreter.get_input_details()[0]
        interpreter.resize_tensor_input(
            input_detail["index"], batch.shape, strict=False
        )
        interpreter.allocate_tensors()
        input_detail = interpreter.get_input_details()[0]
        output_detail = interpreter.get_output_details()[0]
        interpreter.set_tensor(
            input_detail["index"], quantize_tflite_input(batch, input_detail)
        )
        interpreter.invoke()
        output = interpreter.get_tensor(output_detail["index"])
        predictions.append(np.argmax(output, axis=1))

    return np.concatenate(predictions)

## 7. Check one prediction from each PTQ path

In [22]:
sample_image = test_images[:1]
sample_label = int(test_labels[0])
fp32_sample_prediction = int(
    predict_keras_labels(model, sample_image, batch_size=1)[0]
)
builtin_sample_prediction = int(
    predict_tflite_labels(BUILTIN_MODEL_PATH, sample_image, batch_size=1)[0]
)
custom_sample_prediction = int(
    predict_keras_labels(custom_model, sample_image, batch_size=1)[0]
)

prediction_table = pd.DataFrame([
    {
        "ptq": "Original FP32 model",
        "true_class": CIFAR10_CLASS_NAMES[sample_label],
        "predicted_class": CIFAR10_CLASS_NAMES[fp32_sample_prediction],
        "correct": fp32_sample_prediction == sample_label,
    },
    {
        "ptq": "Built-in full INT8",
        "true_class": CIFAR10_CLASS_NAMES[sample_label],
        "predicted_class": CIFAR10_CLASS_NAMES[builtin_sample_prediction],
        "correct": builtin_sample_prediction == sample_label,
    },
    {
        "ptq": "Custom per-channel INT8 weights",
        "true_class": CIFAR10_CLASS_NAMES[sample_label],
        "predicted_class": CIFAR10_CLASS_NAMES[custom_sample_prediction],
        "correct": custom_sample_prediction == sample_label,
    },
])
prediction_table

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,ptq,true_class,predicted_class,correct
0,Original FP32 model,cat,cat,True
1,Built-in full INT8,cat,cat,True
2,Custom per-channel INT8 weights,cat,cat,True


## 8. Evaluate the original model and both PTQ paths

All three rows use the same test images, labels, preprocessing, and trained checkpoint. The original FP32 model is the accuracy baseline, so its drop from FP32 is zero.

In [23]:
fp32_predictions = predict_keras_labels(model, test_images)
custom_predictions = predict_keras_labels(custom_model, test_images)
builtin_predictions = predict_tflite_labels(BUILTIN_MODEL_PATH, test_images)

fp32_accuracy = float(np.mean(fp32_predictions == test_labels))
custom_accuracy = float(np.mean(custom_predictions == test_labels))
builtin_accuracy = float(np.mean(builtin_predictions == test_labels))

accuracy_table = pd.DataFrame([
    {
        "ptq": "Original FP32 model",
        "quantization": "FP32 Keras baseline",
        "correct_predictions": int(np.sum(fp32_predictions == test_labels)),
        "test_images": len(test_labels),
        "accuracy_percent": fp32_accuracy * 100,
        "drop_from_fp32_percentage_points": 0.0,
    },
    {
        "ptq": "Built-in full INT8",
        "quantization": "W8A8, executable TFLite",
        "correct_predictions": int(np.sum(builtin_predictions == test_labels)),
        "test_images": len(test_labels),
        "accuracy_percent": builtin_accuracy * 100,
        "drop_from_fp32_percentage_points": (fp32_accuracy - builtin_accuracy) * 100,
    },
    {
        "ptq": "Custom per-channel INT8 weights",
        "quantization": "W8 weights; FP32 Keras evaluation",
        "correct_predictions": int(np.sum(custom_predictions == test_labels)),
        "test_images": len(test_labels),
        "accuracy_percent": custom_accuracy * 100,
        "drop_from_fp32_percentage_points": (fp32_accuracy - custom_accuracy) * 100,
    },
])

print(f"FP32 reference accuracy: {fp32_accuracy:.2%}")

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


FP32 reference accuracy: 90.47%


## 9. Save results

In [24]:
accuracy_csv_path = OUTPUT_DIR / "ptq_accuracy_comparison.csv"
tensor_csv_path = OUTPUT_DIR / "ptq_tensor_analysis.csv"
memory_csv_path = OUTPUT_DIR / "ptq_parameter_memory_analysis.csv"
results_json_path = OUTPUT_DIR / "results.json"
accuracy_table.to_csv(accuracy_csv_path, index=False)
tensor_analysis_table.to_csv(tensor_csv_path, index=False)
memory_analysis_table.to_csv(memory_csv_path, index=False)
results_json_path.write_text(
    json.dumps(
        {
            "trained_model": str(MODEL_PATH),
            "evaluation_images": int(len(test_labels)),
            "calibration_images": int(len(calibration_samples)),
            "fp32_accuracy_percent": fp32_accuracy * 100,
            "builtin_full_int8_accuracy_percent": builtin_accuracy * 100,
            "custom_weight_ptq_accuracy_percent": custom_accuracy * 100,
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
print(f"Saved table: {accuracy_csv_path}")
print(f"Saved tensor analysis: {tensor_csv_path}")
print(f"Saved memory analysis: {memory_csv_path}")
print(f"Saved summary: {results_json_path}")

Saved table: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_ptq_accuracy_notebook/ptq_accuracy_comparison.csv
Saved tensor analysis: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_ptq_accuracy_notebook/ptq_tensor_analysis.csv
Saved memory analysis: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_ptq_accuracy_notebook/ptq_parameter_memory_analysis.csv
Saved summary: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_ptq_accuracy_notebook/results.json


## Final PTQ accuracy comparison (%)

In [25]:
final_accuracy_table = accuracy_table.copy()
final_accuracy_table["accuracy_percent"] = final_accuracy_table[
    "accuracy_percent"
].map(lambda value: f"{value:.2f}%")
final_accuracy_table["drop_from_fp32_percentage_points"] = (
    final_accuracy_table["drop_from_fp32_percentage_points"]
    .map(lambda value: f"{value:.2f}")
)
final_accuracy_table

,ptq,quantization,correct_predictions,test_images,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 model,FP32 Keras baseline,9047,10000,90.47%,0.00
1,Built-in full INT8,"W8A8, executable TFLite",8853,10000,88.53%,1.94
2,Custom per-channel INT8 weights,W8 weights; FP32 Keras evaluation,8988,10000,89.88%,0.59
